In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "experiments/amoc"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from experiments.indicators import rolling_variance, rolling_ar1


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

from amoc_utils import get_control_all, plot_grouped_on_axis

import kews
from kews.observables import rff, time_delay, mlp

LABEL_FONTSIZE = 16
YLABEL_FONTSIZE = 12
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13
EWS_MEMBER_ALPHA = 0.50


### Full 10-member SSP245 analysis
Compute early warning indicators (`AR1`, `Var`, `ResDMD`, `ResDMD+control` variants)
using `data/amoc_si_mldmax_2Juan.nc`.

- C1 (tipping): members `a`, `i`
- C0 (non-tipping): all other members
- Curves: individual members (light dashed) + C1/C0 means (solid)

In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_FILE = DATA_DIR / 'amoc_si_mldmax_2Juan.nc'

# State variable for EWS indicators.
STATE_VAR = 'amoc48N_ssp245'  # options: amoc48N_ssp245, amoc26N_ssp245

# Forcing variable for ResDMD with control.
# Options from file: 
#     si_flux_denmarkstr_ssp245, mldmax_labr_ssp245, mldmax_irm_ssp245, mldmax_sco_ssp245, mldmax_gin_ssp245
CONTROL_VAR = 'si_flux_denmarkstr_ssp245'
# Control preprocessing options: 'none' | 'zscore' | 'rolling_mean'
# Rolling mean uses CONTROL_ROLLING_WINDOW_YEARS.
CONTROL_PREPROCESS = 'rolling_mean'
CONTROL_ROLLING_WINDOW_YEARS = 10

YEAR0 = 2015
WL = 0.2
DELAY_EMBEDDINGS = [5, 10, 20]

MEMBER_LABELS = np.array(list('abcdefghij'))
C1_MEMBERS = {'a', 'i'}
C1_MASK = np.array([m in C1_MEMBERS for m in MEMBER_LABELS], dtype=bool)

ds = xr.open_dataset(DATA_FILE)
years = YEAR0 + ds['time'].values.astype(int)

state_all = np.asarray(ds[STATE_VAR].values, dtype=np.float32)

print('Dataset:', DATA_FILE)
print('State variable:', STATE_VAR)
print('Control variable:', CONTROL_VAR)

print('Control preprocess:', CONTROL_PREPROCESS, '(window=' + str(CONTROL_ROLLING_WINDOW_YEARS) + ' yr)')
print('Members C1:', MEMBER_LABELS[C1_MASK].tolist())
print('Members C0:', MEMBER_LABELS[~C1_MASK].tolist())


In [ ]:
# Plot state variable
STATE_YLABELS = {
    "amoc26N_ssp245": r"AMOC Strength [Sv]",
    "amoc48N_ssp245": r"AMOC Strength [Sv]",
}

fig, axes = plt.subplots(1, 1, figsize=(9, 4), sharex=True)
plot_grouped_on_axis(
    axes,
    years,
    state_all,
    C1_MASK,
    ylabel=STATE_YLABELS.get(STATE_VAR, STATE_VAR),
    log_scale=False,
)
axes.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/amoc_{STATE_VAR}_traj.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Plot control variable
control_all = get_control_all(
    ds,
    CONTROL_VAR,
    preprocess=CONTROL_PREPROCESS,
    rolling_window=CONTROL_ROLLING_WINDOW_YEARS,
)

CONTROL_YLABELS = {
    "si_flux_denmarkstr_ssp245": "Denmark Straits sea-ice flux [kg/s]",
    "mldmax_labr_ssp245": "Labrador Sea MLD max",
    "mldmax_irm_ssp245": "Irminger Sea MLD max",
    "mldmax_sco_ssp245": "Scotland-Iceland Basin MLD max",
    "mldmax_gin_ssp245": "Greenland-Iceland-Norway Seas MLD max",
}

fig, axes = plt.subplots(1, 1, figsize=(9, 4), sharex=True)
plot_grouped_on_axis(
    axes,
    years,
    control_all,
    C1_MASK,
    ylabel=CONTROL_YLABELS.get(CONTROL_VAR, CONTROL_VAR),
    log_scale=False,
)
axes.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/amoc_{CONTROL_VAR}_traj.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
wL = 0.2

def compute_ews_metrics(trajectories, wL, controls=None, control_method="augmented", augmented_future_input="future"):
    """Compute EWS metrics for a list of trajectories.

    If controls is provided, it must be a list of 1D arrays (one per trajectory),
    each of length cutoff_idx. ResDMD with control is then computed and returned
    as 'resdmd_control_series' and 'resdmd_control_mlp_series'.
    """
    ar1_series = []
    var_series = []
    resdmd_series = []
    resdmd_control_series = [] if controls is not None else None
    resdmd_control_mlp_series = [] if controls is not None else None

    for i, traj in enumerate(trajectories):
        data = np.asarray(traj)

        # AR1, variance, ResDMD
        ar1 = rolling_ar1(data, window=wL)
        var = rolling_variance(data, window=wL)
        resdmd = kews.resdmd(data, window=wL, observable=time_delay(n_delay=20))

        ar1_series.append(np.array(ar1))
        var_series.append(np.array(var))
        resdmd_series.append(np.array(resdmd))

        # ResDMD with control
        if controls is not None:
            u = np.asarray(controls[i])
            resdmd_c = kews.resdmd(data, 
                window=wL, control=u, 
                observable=time_delay(n_delay=20), 
                control_method=control_method, future_input=augmented_future_input
            )

            resdmd_c_mlp = kews.resdmd(data, 
                window=wL, control=u,
                observable=mlp(hidden_layers=[16, 8], bottleneck_dim=4),
                fit_kwargs=dict(
                    lr=1e-3,
                    epochs=1000,
                    batch_size=32,
                    shuffle=True
                ),
                control_method=control_method, future_input=augmented_future_input
            )
            
            resdmd_control_series.append(np.array(resdmd_c))
            resdmd_control_mlp_series.append(np.array(resdmd_c_mlp))

    out = {
        'ar1_series': ar1_series,
        'var_series': var_series,
        'resdmd_series': resdmd_series,
    }
    if resdmd_control_series is not None:
        out['resdmd_control_series'] = resdmd_control_series
        out['resdmd_control_mlp_series'] = resdmd_control_mlp_series
    return out

# Compute EWS (trajectories are already window [start_ews:cutoff_idx] of length num_timesteps)
tipping_ews = compute_ews_metrics(state_all[C1_MASK], wL, controls=control_all[C1_MASK], control_method="augmented", augmented_future_input="future")
non_tipping_ews = compute_ews_metrics(state_all[~C1_MASK], wL, controls=control_all[~C1_MASK], control_method="augmented", augmented_future_input="future")


In [ ]:
# Plot EWS
start_idx = 0
t = np.arange(len(tipping_ews["var_series"][0])) + YEAR0

def masked_array(series_list, start_idx):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    return arr

def plot_distribution(ax, series_list, color, label, start_idx):
    arr = masked_array(series_list, start_idx)
    for i in range(arr.shape[0]):
        ax.plot(t, arr[i], "--", color=color, linewidth=1.5, alpha=EWS_MEMBER_ALPHA)
    median = np.nanmedian(arr, axis=0)
    ax.plot(t, median, color=color, label=label, linewidth=2)

def robust_ylim(series_lists, start_idx, low=2, high=98, margin=0.05):
    vals = np.concatenate([masked_array(s, start_idx)[:, start_idx:].ravel() for s in series_lists])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharex=True)

panel_specs = [
    ("ar1_series", "AR1", False),
    ("var_series", "Variance", False),
    ("resdmd_control_series", "ResKMDc (TDF)", False),
    ("resdmd_control_mlp_series", "ResKMDc (MLP)", False),
]

for ax, (key, ylabel, use_robust_ylim) in zip(axes, panel_specs):
    plot_distribution(ax, tipping_ews[key], "C3", "Tipping", start_idx)
    plot_distribution(ax, non_tipping_ews[key], "C0", "Non-tipping", start_idx)

    ax.set_xlabel("Year", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=YLABEL_FONTSIZE)
    ax.set_xlim([t[0], t[-1]])
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

    if use_robust_ylim:
        ax.set_ylim(robust_ylim([tipping_ews[key], non_tipping_ews[key]], start_idx))

axes[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/amoc_{STATE_VAR}_ews.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# AUROC benchmark across trajectories
import pandas as pd
from experiments.metrics import ensemble_roc, plot_auroc, plot_roc

y_true = np.r_[np.ones(len(tipping_ews['ar1_series'])), np.zeros(len(non_tipping_ews['ar1_series']))].astype(int)

result = ensemble_roc(
    y_true,
    {
        'AR1': tipping_ews['ar1_series'] + non_tipping_ews['ar1_series'],
        'Variance': tipping_ews['var_series'] + non_tipping_ews['var_series'],
        'ResKMDc (TDF)': tipping_ews['resdmd_control_series'] + non_tipping_ews['resdmd_control_series'],
        'ResKMDc (MLP)': tipping_ews['resdmd_control_mlp_series'] + non_tipping_ews['resdmd_control_mlp_series'],
    },
)

auroc_df = pd.DataFrame(
    [
        {
            'indicator': name,
            'auroc': curve.auroc,
            'n_valid': curve.n_valid,
            'n_missing': curve.n_missing,
        }
        for name, curve in result.curves.items()
    ]
).sort_values('auroc', ascending=False).reset_index(drop=True)
display(auroc_df)

fig, ax = plt.subplots(figsize=(6, 4))
plot_auroc(result, ax=ax, label_fontsize=LABEL_FONTSIZE, tick_labelsize=TICK_FONTSIZE)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/amoc_{STATE_VAR}_auroc.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(
    result,
    ax=ax,
    title=None,
    chance=True,
    show_auroc=False,
    linewidth=3,
    bold_top_n=2,
    label_fontsize=LABEL_FONTSIZE,
    tick_labelsize=TICK_FONTSIZE,
    legend_fontsize=LEGEND_FONTSIZE,
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/amoc_{STATE_VAR}_roc.png", dpi=300, bbox_inches="tight")
plt.show()
